In [ ]:
# This script implements teacher forcing, i.e. sometimes feeding real data and sometimes feeding the generated data for training.

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle as pkl
import random
from model import SequenceGenerator

# DATA LOADING
DATASETS_PATH = os.path.join('..', '..', '..','data')
TEST_DATASET_PATH = os.path.join(DATASETS_PATH, 'test.pickle')
TRAIN_DATASET_PATH = os.path.join(DATASETS_PATH, 'train.pickle')

with open(TEST_DATASET_PATH, 'rb') as f:
    test_dataset = pkl.load(f)
    print(test_dataset['label'].value_counts())

with open(TRAIN_DATASET_PATH, 'rb') as f:
    train_dataset = pkl.load(f)
    print(train_dataset['label'].value_counts())

# DATASET CLASS
class SensorDataSet(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __getitem__(self, idx):
        data = self.dataset.iloc[idx]
        x = torch.tensor(data['sensor_data'], dtype=torch.float32)
        y = torch.tensor(data['label'], dtype=torch.long)
        return x, y

    def __len__(self):
        return len(self.dataset)


# TRAINING FUNCTION
def train_generator(generator, train_loader, num_epochs=10, lr=0.001, teacher_forcing_start=1.0, teacher_forcing_end=0.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    generator.to(device)
    optimizer = torch.optim.Adam(generator.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    generator.train()
    total_steps = len(train_loader) * num_epochs
    step = 0

    for epoch in range(num_epochs):
        total_loss = 0
        for x, _ in train_loader:
            x = x.to(device)  # (batch, 128, 6)
            batch_size, seq_len, input_dim = x.size()

            # Input for first timestep
            input_t = x[:, 0, :].unsqueeze(1)  # first timestep: (batch, 1, 6)

            # Scheduled teacher forcing rate: how much real data gets fed for this timestep?
            teacher_forcing_ratio = teacher_forcing_end + (teacher_forcing_start - teacher_forcing_end) * np.exp(-5 * step / total_steps)
            step += 1

            hidden = None
            outputs = []

            # Iterate through sequence timestep by timestep and feed real values x or generated values output as the next input according to the current teacher forcing rate
            for t in range(1, seq_len):
                output, hidden = generator(input_t, hidden)  # output: (batch, 1, 6)
                outputs.append(output)

                use_teacher = torch.rand(batch_size, device=device) < teacher_forcing_ratio # determine who out of the batch gets real and who gets generated data for this timestep
                use_teacher = use_teacher.unsqueeze(1).unsqueeze(2)  # (batch, 1, 1)

                next_input = use_teacher * x[:, t, :].unsqueeze(1) + (~use_teacher) * output.detach()
                input_t = next_input

            # Output and target for whole sequence
            outputs = torch.cat(outputs, dim=1)  # (batch, seq_len - 1, 6)
            target = x[:, 1:, :]  # (batch, seq_len - 1, 6)

            # Loss between output and target
            loss = loss_fn(outputs, target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * x.size(0)

        # Since the teacher forcing rate decreases (exponentially) over time we expect the loss to potentially increase and decrease multiple times.
        # First the model learns what the desired sequence is on the real data and as more and more generated data gets fed it has to relearn how it can achieve such sequences given its own output.
        print(f"[Epoch {epoch+1}] Loss: {total_loss / len(train_loader.dataset):.6f}")

# INITIALIZING AND TRAINING
input_dim = 6
hidden_dim = 128
num_layers = 2

generator = SequenceGenerator(input_dim, hidden_dim, num_layers)

train_loader = DataLoader(SensorDataSet(train_dataset), batch_size=32, shuffle=True)

train_generator(generator, train_loader, num_epochs=10, lr=0.001)


In [ ]:
# SEQUENCE GENERATION FUNCTION
def generate_sequence(generator, seed, seq_len=128):
    generator.eval()
    device = next(generator.parameters()).device

    # Format starting seed as (1, 1, 6) tensor, because the LSTM wants those dimensions
    seed = torch.tensor(seed, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    generated = [seed.squeeze(0)] # Store first sequence step
    hidden = None

    # Regressively generate a sequence by using last output as next input
    for _ in range(seq_len - 1):
        output, hidden = generator(seed, hidden)
        seed = output
        generated.append(output.squeeze(0))

    return torch.cat(generated, dim=0).cpu().detach() # generated sequence: (128, 6)

# PLOTTING FUNCTION
def plot_sequence(dataset, return_figure=True):
    fig, axes = plt.subplots(3,2)

    directions = ["x", "y", "z"]

    value_type = ['acceleration', 'rotation']

    for i, direction_label in enumerate(directions):
        for j, value_label in enumerate(value_type):
            axes[i,j].plot(dataset[...,:,3*j+i])
            if i == 0:
                axes[i,j].set_title(value_label)
                
            if j == 1:
                axes[i,j].text(1, 0.5,direction_label, size=12, rotation=270, transform=axes[i,j].transAxes)
    if return_figure:
        return fig

# PLOT 3 SYNTHETIC SEQUENCES
# Collect mean and std of real starting values for generating a new random starting value
start_values = np.stack(train_dataset['sensor_data'].apply(lambda x: x[0]))  # shape: (num_samples, 6)
mean_start = start_values.mean(axis=0)  # (6,)
std_start = start_values.std(axis=0)    # (6,)
std_factor = 0.5

# Generate sequence
for i in range(3):
    # Random starting seed (starting value)
    seed = mean_start + np.random.normal(0, std_start*std_factor, size=mean_start.shape)

    fake_seq = generate_sequence(generator, seed)
    plot_sequence(fake_seq.numpy())